## GitHub Sample: Faker Dataset + API Validation
Intent: Generate a small reproducible sample for GitHub to run independently without needing the full dataset.
Method: Uses Python Faker to generate 20 synthetic US addresses with controlled mutations (typo, ZIP mismatch, missing unit). Calls the Precisely Address Verification API on all 20. Assigns PBKEY only to successfully verified records. Exports 10-row sample_before.csv (no PBKEY) and sample_after.csv (PBKEY where verified) to data/.
Why: Demonstrates the before/after concept in a self-contained, runnable form without exposing the full 250-record dataset.

In [ ]:
# ============================================================
# Sample Data Generator for GitHub
# Generates 20 synthetic records, validates via Precisely API,
# assigns PBKEY to successful matches, exports 10-row samples
# ============================================================

import os
import re
import random
import requests
import pandas as pd
from pathlib import Path
from faker import Faker
from dotenv import load_dotenv
from pathlib import Path


load_dotenv()
fake = Faker("en_US")
random.seed(42)

# ── Auth ─────────────────────────────────────────────────────
BASE_URL    = os.getenv("PRECISELY_BASE_URL")
API_KEY     = os.getenv("PRECISELY_API_KEY")
API_SECRET  = os.getenv("PRECISELY_API_SECRET")

def get_token():
    r = requests.post(
        f"{BASE_URL}/oauth/token",
        data={"grant_type": "client_credentials"},
        auth=(API_KEY, API_SECRET)
    )
    r.raise_for_status()
    return r.json()["access_token"]

TOKEN = get_token()
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

# ── US States lookup ─────────────────────────────────────────
US_STATES = [
    "AL","AZ","AR","CA","CO","CT","DE","FL","GA","ID",
    "IL","IN","IA","KS","KY","LA","ME","MD","MA","MI",
    "MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY",
    "NC","ND","OH","OK","OR","PA","RI","SC","SD","TN",
    "TX","UT","VT","VA","WA","WV","WI","WY"
]

PROP_TYPES  = ["R", "B", "M", "X"]
LANG_CODES  = ["ENG"]
STREET_SFXS = ["St", "Ave", "Blvd", "Dr", "Ln", "Rd", "Way", "Ct"]

# ── Mutation helpers ─────────────────────────────────────────
def typo_street(name):
    """Transpose two adjacent characters in street name."""
    if len(name) < 4:
        return name
    i = random.randint(1, len(name) - 2)
    lst = list(name)
    lst[i], lst[i+1] = lst[i+1], lst[i]
    return "".join(lst)

def zip_city_mismatch(zip_code):
    """Increment ZIP by a random offset to create mismatch."""
    try:
        return str(int(zip_code) + random.randint(10, 99)).zfill(5)
    except ValueError:
        return zip_code

# ── Generate 20 synthetic records ────────────────────────────
DIRTY_INDICES = random.sample(range(20), 6)   # 6 dirty, 14 clean

records = []
for i in range(20):
    street_num  = str(random.randint(100, 9999))
    street_name = fake.last_name()             # realistic street name
    suffix      = random.choice(STREET_SFXS)
    city        = fake.city()
    state       = random.choice(US_STATES)
    zip_code    = fake.zipcode()
    unit        = f"Apt {random.randint(1,200)}" if random.random() < 0.3 else ""
    prop_type   = random.choice(PROP_TYPES)
    fips        = fake.numerify(text="#####")

    is_dirty          = i in DIRTY_INDICES
    mutation_category = ""
    mutation_desc     = ""

    if is_dirty:
        mutation_type = random.choice(["typo", "zip_mismatch", "missing_unit"])
        if mutation_type == "typo":
            street_name       = typo_street(street_name)
            mutation_category = "typographic"
            mutation_desc     = "Transposed characters in street name"
        elif mutation_type == "zip_mismatch":
            zip_code          = zip_city_mismatch(zip_code)
            mutation_category = "zip_city_mismatch"
            mutation_desc     = "ZIP code does not match city"
        elif mutation_type == "missing_unit":
            unit              = ""
            mutation_category = "structural"
            mutation_desc     = "Missing unit/apt number"

    addr_line1      = f"{street_num} {street_name} {suffix}"
    formatted_addr  = f"{addr_line1}{', ' + unit if unit else ''}, {city}, {state} {zip_code}"

    records.append({
        "FORMATTEDADDRESS" : formatted_addr,
        "ADDRLINE1"        : addr_line1,
        "ADDRLINE2"        : unit,
        "ADDRLINE3"        : "",
        "ADDRLINE4"        : "",
        "ADDNUMBER"        : street_num,
        "ADDNUMBER2"       : "",
        "STREETNAME"       : street_name,
        "STREETPREDIR"     : "",
        "STREET"           : street_name,
        "STREETSUFFIX"     : suffix,
        "STREETPOSTDIR"    : "",
        "UNITTYPE"         : "Apt" if unit else "",
        "UNIT"             : unit.replace("Apt ", "") if unit else "",
        "LEVEL"            : "",
        "BUILDINGNAME"     : "",
        "CITY"             : city,
        "POSTALCODE"       : zip_code,
        "POSTALCODEEXT"    : "",
        "ADMIN2"           : fake.city(),
        "ADMIN1"           : state,
        "COUNTRY"          : "USA",
        "LOCCODE"          : "",
        "LATITUDE"         : "",
        "LONGITUDE"        : "",
        "PROPTYPE"         : prop_type,
        "LANGCODE"         : "ENG",
        "FIPS"             : fips,
        "is_dirty"         : is_dirty,
        "mutation_category": mutation_category,
        "mutation_desc"    : mutation_desc,
    })

df_before = pd.DataFrame(records)
print(f"✅ Generated {len(df_before)} records — {df_before['is_dirty'].sum()} dirty, {(~df_before['is_dirty']).sum()} clean")

# ── Validate via Precisely API ────────────────────────────────
VERIFY_URL = f"{BASE_URL}/address/v1/transient/verify"

def verify_address(row):
    payload = {
        "Input": {
            "Row": [{
                "AddressLine1" : row["ADDRLINE1"],
                "AddressLine2" : row["ADDRLINE2"],
                "City"         : row["CITY"],
                "StateProvince": row["ADMIN1"],
                "PostalCode"   : row["POSTALCODE"],
                "Country"      : "USA"
            }]
        }
    }
    try:
        r = requests.post(VERIFY_URL, json=payload, headers=HEADERS, timeout=10)
        r.raise_for_status()
        result = r.json()["Output"][0]
        return {
            "mcp_verify_status"  : result.get("Status", ""),
            "mcp_verified_address": result.get("AddressLine1", ""),
            "mcp_verified_city"  : result.get("City", ""),
            "mcp_verified_state" : result.get("StateProvince", ""),
            "mcp_verified_zip"   : result.get("PostalCode", ""),
            "mcp_verify_score"   : result.get("MatchScore", 0),
            "mcp_street_corrected": result.get("AddressLine1", "") != row["ADDRLINE1"],
            "mcp_zip_corrected"  : result.get("PostalCode", "") != row["POSTALCODE"],
            "mcp_city_corrected" : result.get("City", "") != row["CITY"],
        }
    except Exception as e:
        return {
            "mcp_verify_status"  : "error",
            "mcp_verified_address": "",
            "mcp_verified_city"  : "",
            "mcp_verified_state" : "",
            "mcp_verified_zip"   : "",
            "mcp_verify_score"   : 0,
            "mcp_street_corrected": False,
            "mcp_zip_corrected"  : False,
            "mcp_city_corrected" : False,
        }

print("🔄 Running address verification on 20 records...")
verify_results = df_before.apply(verify_address, axis=1, result_type="expand")
df_after = pd.concat([df_before, verify_results], axis=1)
print(f"✅ Verification complete — {(df_after['mcp_verify_status'] == 'V').sum()} verified")

# ── Assign PBKEY only to verified records ─────────────────────
def assign_pbkey(row):
    if row["mcp_verify_status"] == "V":
        return "PB" + fake.bothify(text="??########").upper()
    return ""

df_after["PBKEY"] = df_after.apply(assign_pbkey, axis=1)
print(f"🔑 PBKEY assigned to {(df_after['PBKEY'] != '').sum()} records")

# ── Export 10-row samples ─────────────────────────────────────
# Select top 10 rows that best represent variety
# Priority: include at least 3 dirty and 7 clean
dirty_rows = df_after[df_after["is_dirty"]].head(3)
clean_rows = df_after[~df_after["is_dirty"]].head(7)
df_sample  = pd.concat([clean_rows, dirty_rows]).reset_index(drop=True)

# Before — drop PBKEY and all mcp/after columns
before_cols = [
    "FORMATTEDADDRESS","ADDRLINE1","ADDRLINE2","ADDRLINE3","ADDRLINE4",
    "ADDNUMBER","ADDNUMBER2","STREETNAME","STREETPREDIR","STREET",
    "STREETSUFFIX","STREETPOSTDIR","UNITTYPE","UNIT","LEVEL","BUILDINGNAME",
    "CITY","POSTALCODE","POSTALCODEEXT","ADMIN2","ADMIN1","COUNTRY",
    "LOCCODE","LATITUDE","LONGITUDE","PROPTYPE","LANGCODE","FIPS",
    "is_dirty","mutation_category","mutation_desc"
]

after_cols = [
    "PBKEY","FORMATTEDADDRESS","ADDRLINE1","ADDRLINE2","CITY","ADMIN1",
    "POSTALCODE","PROPTYPE","FIPS","LATITUDE","LONGITUDE",
    "is_dirty","mutation_category","mutation_desc",
    "mcp_verify_status","mcp_verified_address","mcp_verified_city",
    "mcp_verified_state","mcp_verified_zip","mcp_verify_score",
    "mcp_street_corrected","mcp_zip_corrected","mcp_city_corrected"
]

BASE_DIR = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path().resolve().parent
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

df_sample[before_cols].to_csv(DATA_DIR / "sample_before.csv", index=False)
df_sample[after_cols].to_csv(DATA_DIR / "sample_after.csv",  index=False)

print("\n📁 Exported:")
print(f"   data/sample_before.csv — {len(df_sample)} rows, {len(before_cols)} columns, no PBKEY")
print(f"   data/sample_after.csv  — {len(df_sample)} rows, {len(after_cols)} columns, PBKEY where verified")
print(f"\n   Dirty records in sample : {df_sample['is_dirty'].sum()}")
print(f"   Clean records in sample : {(~df_sample['is_dirty']).sum()}")
print(f"   PBKEY populated         : {(df_sample['PBKEY'] != '').sum()}")

## Dependencies
Intent: Ensure all required Python packages are installed into the active notebook kernel.
Method: Uses sys.executable to install from requirements.txt into the same Python environment the kernel is running, confirming the interpreter path.
Why: Avoids module not found errors when running the notebook fresh without a pre-activated virtual environment.

In [ ]:
import sys
print(f"Python interpreter: {sys.executable}")
!{sys.executable} -m pip install -r requirements.txt
print("✅ Dependencies ready")

## Configuration
Intent: Define all constants, file paths, and model settings used throughout the notebook in one place.
Method: Sets BASE_DIR, DATA_DIR, SEED_FILE, OUT_FILE using pathlib.Path. Configures Ollama model name, mutation budget (50 records), and random seed for reproducibility.

Why: Centralizing configuration makes the notebook portable and easy to adapt without hunting through code cells.

In [1]:
import pandas as pd
import random
import json
import requests
import re
from pathlib import Path

import os
from dotenv import load_dotenv

load_dotenv()

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR   = Path().resolve().parent   # notebooks/ → project root
DATA_DIR   = BASE_DIR / "data"
SEED_FILE  = DATA_DIR / "rawdata.csv"
OUT_FILE   = DATA_DIR / "synthetic_250_restapi.csv"

# ── Precisely API config ───────────────────────────────────────────────────
PRECISELY_API_KEY    = os.getenv("PRECISELY_API_KEY")
PRECISELY_API_SECRET = os.getenv("PRECISELY_API_SECRET")
PRECISELY_TOKEN_URL  = "https://api.precisely.com/oauth/token"

# __ MCP Server URL
MCP_BASE_URL = "http://127.0.0.1:8000/mcp"

# ── Ollama config ──────────────────────────────────────────────────────────
OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "mistral:7b"

# ── Records that will be tainted  ────────────────────────────────────────────────────────
TOTAL_DIRTY      = 55
RANDOM_SEED      = 42          # reproducibility
random.seed(RANDOM_SEED)

# ── Tainted data distribution (must sum to TOTAL_DIRTY) ────────────────────────
MUTATION_BUDGET = {
    "flood_zip_mismatch"   : 15,   # M, R coastal → swap to inland ZIP
    "missing_unit"         : 12,   # M, B → drop UNITTYPE + UNIT
    "street_typo"          : 10,   # R, B, M → character-level noise
    "zip_city_mismatch"    : 13,   # R, X → valid but mismatched pair
    "unresolvable"         :  5,   # X, R → blank ADDNUMBER, corrupt street
}

assert sum(MUTATION_BUDGET.values()) == TOTAL_DIRTY, "Budget must sum to 50"

print("✅ Config loaded")
print(f"   Seed file : {SEED_FILE}")
print(f"   Output    : {OUT_FILE}")
print(f"   Model     : {OLLAMA_MODEL}")
print(f"   Dirty     : {TOTAL_DIRTY} / 250 records")

✅ Config loaded
   Seed file : /Users/senthilpanchatcharam/projects/address-intelligence-prototype/data/rawdata.csv
   Output    : /Users/senthilpanchatcharam/projects/address-intelligence-prototype/data/synthetic_250_restapi.csv
   Model     : mistral:7b
   Dirty     : 55 / 250 records


## Load Seed Data (Real Addresses)
Intent: Load 250 real, Precisely-verified addresses as the ground truth dataset.
Method: Reads rawdata.csv — addresses sourced from Precisely's own dataset, already geocoded and verified. Confirms schema completeness and field coverage.

Why: Using real verified addresses as the base ensures Precisely APIs return meaningful data during enrichment. Faker-generated addresses would fail API verification and enrichment pipeline.

In [ ]:
df = pd.read_csv(SEED_FILE, dtype=str).fillna("")

# ── Schema check ───────────────────────────────────────────────────────────
REQUIRED_COLS = [
    "PBKEY","PARENT","GOVID","GEOID","FORMATTEDADDRESS",
    "ADDRLINE1","ADDRLINE2","ADDRLINE3","ADDRLINE4",
    "ADDNUMBER","ADDNUMBER2","STREETNAME","STREETPREDIR",
    "STREET","STREETSUFFIX","STREETPOSTDIR","UNITTYPE","UNIT",
    "LEVEL","BUILDINGNAME","CITY","POSTALCODE","POSTALCODEEXT",
    "ADMIN2","ADMIN1","COUNTRY","LOCCODE","LATITUDE","LONGITUDE",
    "PROPTYPE","LANGCODE","FIPS"
]

missing = [c for c in REQUIRED_COLS if c not in df.columns]
assert not missing, f"Missing columns: {missing}"

print(f"✅ Loaded {len(df)} records — schema OK")
print(f"\nPROPTYPE distribution:")
print(df["PROPTYPE"].value_counts().to_string())
print(f"\nField completeness (non-empty %):")
key_fields = ["ADDNUMBER","UNITTYPE","UNIT","CITY","POSTALCODE","STREETNAME"]
for col in key_fields:
    pct = (df[col] != "").mean() * 100
    print(f"  {col:<20} {pct:.1f}%")

## Identify Records to be Tainted/Dirty
Intent: Deterministically select 50 records from the 250 to receive controlled errors.
Method: Uses random.seed(42) for reproducibility. Selects parameterized (55) records weighted by PROPTYPE to ensure variety across residential, commercial, and mixed-use properties.
Why: Seeded selection means the same 50 records are chosen every run — results are reproducible and not random between sessions.

In [ ]:
selected = {}

# ── Helper: sample from filtered pool without replacement ──────────────────
def sample_pool(mask, n, label, exclude=set()):
    pool = df[mask & ~df.index.isin(exclude)].index.tolist()
    if len(pool) < n:
        print(f"⚠️  {label}: requested {n}, only {len(pool)} available — using all")
        n = len(pool)
    return set(random.sample(pool, n))

used = set()

# 1. flood_zip_mismatch → M and coastal R (ADMIN1=NJ, prioritise M first)
m_flood   = sample_pool(df["PROPTYPE"].isin(["M"]), min(7, MUTATION_BUDGET["flood_zip_mismatch"]), "flood/M")
r_flood   = sample_pool(df["PROPTYPE"] == "R", MUTATION_BUDGET["flood_zip_mismatch"] - len(m_flood), "flood/R", exclude=used)
selected["flood_zip_mismatch"] = m_flood | r_flood
used |= selected["flood_zip_mismatch"]

# 2. missing_unit → B first, then R (M pool exhausted by flood step)
has_unit  = df["UNIT"] != ""
b_unit    = sample_pool((df["PROPTYPE"] == "B") & has_unit, min(6, MUTATION_BUDGET["missing_unit"]), "unit/B", exclude=used)
r_unit    = sample_pool((df["PROPTYPE"] == "R") & has_unit, MUTATION_BUDGET["missing_unit"] - len(b_unit), "unit/R", exclude=used)
m_unit    = set()
selected["missing_unit"] = m_unit | b_unit | r_unit
used |= selected["missing_unit"]

# 3. street_typo → R, B, M
selected["street_typo"] = sample_pool(df["PROPTYPE"].isin(["R","B","M"]), MUTATION_BUDGET["street_typo"], "typo", exclude=used)
used |= selected["street_typo"]

# 4. zip_city_mismatch → R and X
selected["zip_city_mismatch"] = sample_pool(df["PROPTYPE"].isin(["R","X"]), MUTATION_BUDGET["zip_city_mismatch"], "zip_city", exclude=used)
used |= selected["zip_city_mismatch"]

# 5. unresolvable → X first, then R
x_unres = sample_pool(df["PROPTYPE"] == "X", min(MUTATION_BUDGET["unresolvable"], 5), "unres/X", exclude=used)
r_unres = sample_pool(df["PROPTYPE"] == "R",  MUTATION_BUDGET["unresolvable"] - len(x_unres), "unres/R", exclude=used)
selected["unresolvable"] = x_unres | r_unres
used |= selected["unresolvable"]

# ── Manifest ───────────────────────────────────────────────────────────────
print(f"✅ Mutation manifest — {len(used)} records selected\n")
for cat, idx in selected.items():
    print(f"  {cat:<25} {len(idx):>3} records")

assert len(used) == TOTAL_DIRTY, f"Expected {TOTAL_DIRTY}, got {len(used)}"
print(f"\n  Total dirty                 {len(used)}")

## Define Functions to introduce errors
Intent: Define one function per mutation category — no data is modified in this cell.
Method: Defines five mutation functions: street name typo (character transposition), missing unit number, ZIP/city mismatch (adjacent real ZIP), outdated street name, and unresolvable address (vowel stripping + blank address number). Each function takes a record and returns a mutated copy with mutation_category and mutation_description fields populated.

Why: Separating function definitions from execution keeps the pipeline clean and each mutation testable independently.

In [ ]:
import re

# ── NJ/NYC ZIP↔CITY swap table (valid pairs, intentionally mismatched) ─────
ZIP_CITY_SWAPS = [
    ("07030", "Hoboken",       "07302", "Jersey City"),
    ("07302", "Jersey City",   "07030", "Hoboken"),
    ("07070", "Rutherford",    "07071", "Lyndhurst"),
    ("07071", "Lyndhurst",     "07070", "Rutherford"),
    ("07601", "Hackensack",    "07650", "Palisades Park"),
    ("07650", "Palisades Park","07601", "Hackensack"),
    ("10001", "New York",      "10002", "New York"),
    ("07047", "North Bergen",  "07093", "West New York"),
    ("07093", "West New York", "07047", "North Bergen"),
    ("07306", "Jersey City",   "07307", "Jersey City"),
]

# Inland ZIPs to swap coastal records into (flood mismatch)
INLAND_ZIPS = [
    ("07001", "Avenel"),
    ("07002", "Bayonne"),
    ("07003", "Bloomfield"),
    ("07004", "Fairfield"),
    ("07005", "Boonton"),
    ("07006", "Caldwell"),
    ("07008", "Carteret"),
    ("07009", "Cedar Grove"),
    ("07010", "Cliffside Park"),
    ("07011", "Clifton"),
    ("07012", "Clifton"),
    ("07013", "Clifton"),
    ("07014", "Clifton"),
    ("07016", "Cranford"),
]

def mutate_flood_zip(row):
    inland = random.choice(INLAND_ZIPS)
    row["POSTALCODE"] = inland[0]
    row["CITY"]       = inland[1]
    row["mutation_category"]    = "flood_zip_mismatch"
    row["mutation_description"] = f"ZIP swapped to inland {inland[0]} ({inland[1]}) — geocode will place record outside flood zone"
    row["expected_api_outcome"] = "partial_verify"
    return row

def mutate_missing_unit(row):
    original_unit = row["UNIT"]
    original_type = row["UNITTYPE"]
    row["UNIT"]     = ""
    row["UNITTYPE"] = ""
    row["mutation_category"]    = "missing_unit"
    row["mutation_description"] = f"UNIT '{original_unit}' and UNITTYPE '{original_type}' removed — system cannot resolve to specific occupancy"
    row["expected_api_outcome"] = "partial_verify"
    return row

def _typo(s):
    """Introduce a single character-level error into a string."""
    if len(s) < 2:
        return s
    ops = ["swap", "drop", "repeat", "replace"]
    op  = random.choice(ops)
    i   = random.randint(0, len(s) - 1)
    if op == "swap" and i < len(s) - 1:
        lst = list(s)
        lst[i], lst[i+1] = lst[i+1], lst[i]
        return "".join(lst)
    elif op == "drop":
        return s[:i] + s[i+1:]
    elif op == "repeat":
        return s[:i] + s[i] + s[i:]
    else:  # replace with adjacent key
        adjacent = {"A":"S","E":"R","I":"O","O":"I","S":"A","T":"R","R":"T","N":"M","M":"N"}
        c = s[i].upper()
        return s[:i] + adjacent.get(c, s[i]) + s[i+1:]

def mutate_street_typo(row):
    original = row["STREETNAME"]
    dirty    = _typo(original)
    row["STREETNAME"] = dirty
    # Also corrupt suffix occasionally
    if row["STREETSUFFIX"] and random.random() > 0.5:
        suffixes = ["St","Steret","Ave","Avee","Blvd","Blv","Rd","Rdd","Dr","Drv"]
        row["STREETSUFFIX"] = random.choice(suffixes)
    row["mutation_category"]    = "street_typo"
    row["mutation_description"] = f"STREETNAME '{original}' → '{dirty}' — character-level noise introduced"
    row["expected_api_outcome"] = "partial_verify"
    return row

def mutate_zip_city(row):
    original_zip  = row["POSTALCODE"]
    original_city = row["CITY"]
    # Find a swap pair or fall back to random swap table entry
    match = next((s for s in ZIP_CITY_SWAPS if s[0] == original_zip), None)
    if match:
        row["CITY"]       = match[3]
        row["POSTALCODE"] = match[2]
    else:
        swap = random.choice(ZIP_CITY_SWAPS)
        row["CITY"]       = swap[3]
        row["POSTALCODE"] = swap[2]
    row["mutation_category"]    = "zip_city_mismatch"
    row["mutation_description"] = f"ZIP {original_zip}/{original_city} swapped to {row['POSTALCODE']}/{row['CITY']} — valid values, wrong pairing"
    row["expected_api_outcome"] = "fail_verify"
    return row

def mutate_unresolvable(row):
    original_num    = row["ADDNUMBER"]
    original_street = row["STREETNAME"]
    row["ADDNUMBER"]  = ""
    row["STREETNAME"] = re.sub(r'[aeiouAEIOU]', '', original_street)  # strip vowels
    row["mutation_category"]    = "unresolvable"
    row["mutation_description"] = f"ADDNUMBER blanked, STREETNAME vowels stripped '{original_street}' → '{row['STREETNAME']}'"
    row["expected_api_outcome"] = "fail_verify"
    return row

# ── Dispatch map ───────────────────────────────────────────────────────────
MUTATION_FN = {
    "flood_zip_mismatch" : mutate_flood_zip,
    "missing_unit"       : mutate_missing_unit,
    "street_typo"        : mutate_street_typo,
    "zip_city_mismatch"  : mutate_zip_city,
    "unresolvable"       : mutate_unresolvable,
}

print("✅ Mutation functions defined")

## Apply Dirty Function + LLM Annotation
Intent: Apply mutations to the selected records and use a local LLM to reason about what will go wrong — establishing the "before thought" that the AI layer will later resolve.
Method: Applies one mutation per selected record. Then calls Ollama (Mistral 7b running locally) with a structured prompt asking it to predict the Precisely API outcome for each dirty record. The LLM reasons over the address fields — identifying typos, structural gaps, ZIP mismatches — and documents its prediction in llm_outcome_review. A Pydantic-constrained prompt template ensures consistent, parseable output.

LLM Role: This is the first LLM reasoning step in the pipeline. Mistral is not just labeling — it is predicting failure modes from raw address data before any API call is made. This intentionally mirrors what the AI layer will do later, but in reverse: here the LLM reads dirty data and anticipates problems. In AI layer it will read enriched data and surface decisions.

Why: Establishes a documented, reproducible pre-validation intelligence layer. The LLM annotations shows even AI reasoning over bad data produces unreliable output. The contrast it with enriched narrative to prove out the importance of having accurate data.

In [ ]:
import re

# ── Add metadata columns to a clean copy ──────────────────────────────────
df_out = df.copy()
df_out["is_dirty"]             = False
df_out["mutation_category"]    = ""
df_out["mutation_description"] = ""
df_out["expected_api_outcome"] = ""
df_out["llm_outcome_review"]   = ""

# ── Apply deterministic mutations ──────────────────────────────────────────
for cat, indices in selected.items():
    fn = MUTATION_FN[cat]
    for idx in indices:
        row = df_out.loc[idx].to_dict()
        row = fn(row)
        row["is_dirty"] = True
        for col in ["is_dirty","mutation_category","mutation_description",
                    "expected_api_outcome","UNIT","UNITTYPE","STREETNAME",
                    "STREETSUFFIX","ADDNUMBER","POSTALCODE","CITY"]:
            df_out.at[idx, col] = row[col]

print(f"✅ Mutations applied — {df_out['is_dirty'].sum()} dirty records")

# ── Per-record Ollama review ───────────────────────────────────────────────
dirty_idx = df_out[df_out["is_dirty"]].index.tolist()
total     = len(dirty_idx)

print(f"\n🚀 Per-record review — {total} records using {OLLAMA_MODEL}...")

for i, idx in enumerate(dirty_idx):
    row    = df_out.loc[idx]
    prompt = (
        f"Address: {row['FORMATTEDADDRESS']}\n"
        f"Mutation: {row['mutation_category']}\n"
        f"Expected outcome: {row['expected_api_outcome']}\n"
        f"Reply with JSON only: {{\"llm_outcome_review\": \"one sentence why\"}}"
    )
    try:
        resp   = requests.post(
            OLLAMA_URL,
            json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False},
            timeout=30
        )
        raw    = resp.json().get("response", "").strip()
        raw    = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
        start  = raw.find("{")
        end    = raw.rfind("}") + 1
        parsed = json.loads(raw[start:end])
        review = parsed.get("llm_outcome_review", "")
    except Exception as e:
        review = f"parse_error: {e}"

    df_out.at[idx, "llm_outcome_review"] = review

    if (i + 1) % 10 == 0 or (i + 1) == total:
        print(f"  [{i+1}/{total}] done")

print(f"\n✅ Review complete")
print(df_out[df_out["is_dirty"]][
    ["FORMATTEDADDRESS","mutation_category","expected_api_outcome","llm_outcome_review"]
].head(5).to_string())

## Rebuild Address Fields
Intent: Reconstruct FORMATTEDADDRESS and ADDRLINE1 from the dirty records so the Precisely API actually sees the dirty data.
Method: Rebuilds the single-line address string from ADDNUMBER, STREETNAME, STREETSUFFIX, UNITTYPE, UNIT, CITY, ADMIN1, POSTALCODE. Applies only to dirty records. Guards against suffix/prefir duplication by checking if its already embedded in STREETNAME.

Why: Dirty Function modify individual fields. Without rebuilding the formatted address, Precisely's API would still read the original string and never encounter the errors.

Note: This could have be folded into earlier cell. Found out the issue while debugging.

In [ ]:
def rebuild_address(row):
    """Reconstruct FORMATTEDADDRESS and ADDRLINE1 from dirty component fields."""
    num     = row["ADDNUMBER"].strip()
    name    = row["STREETNAME"].strip()

    # Only append predir/suffix/postdir if not already contained in STREETNAME
    predir  = row["STREETPREDIR"].strip()
    suffix  = row["STREETSUFFIX"].strip()
    postdir = row["STREETPOSTDIR"].strip()

    predir  = predir  if predir  and predir  not in name else ""
    suffix  = suffix  if suffix  and suffix  not in name else ""
    postdir = postdir if postdir and postdir not in name else ""

    street   = " ".join(p for p in [num, predir, name, suffix, postdir] if p)

    # Unit
    utype  = row["UNITTYPE"].strip()
    unit   = row["UNIT"].strip()
    unit_str = f"{utype} {unit}".strip() if utype or unit else ""

    # City, state, zip
    city   = row["CITY"].strip()
    state  = row["ADMIN1"].strip()
    zcode  = row["POSTALCODE"].strip()
    zext   = row["POSTALCODEEXT"].strip()
    zip_str= f"{zcode}-{zext}" if zext else zcode

    # ADDRLINE1 = street + unit
    addrline1 = " ".join(p for p in [street, unit_str] if p)

    # FORMATTEDADDRESS = full single line
    formatted = " ".join(p for p in [addrline1, city, state, zip_str] if p)

    return addrline1, formatted

# Apply only to dirty records
for idx in df_out[df_out["is_dirty"]].index:
    addrline1, formatted        = rebuild_address(df_out.loc[idx])
    df_out.at[idx, "ADDRLINE1"]        = addrline1
    df_out.at[idx, "FORMATTEDADDRESS"] = formatted

print("✅ FORMATTEDADDRESS and ADDRLINE1 rebuilt for dirty records")
print(df_out[df_out["is_dirty"]][["ADDRLINE1","FORMATTEDADDRESS"]].head(5).to_string())

## Export Synthetic Dataset
Intent: Merge clean and dirty records into one file for the pipeline.
Method: Concatenates the two subsets, resets the index, and writes to OUT_FILE.csv. Annotates is_dirty boolean and includes all mutation metadata columns.
Why: Single combined file feeds both the REST and MCP pipeline cells that follow, keeping the pipeline linear and traceable.

In [ ]:
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(OUT_FILE, index=False)

total_records = len(df_out)
dirty_count   = df_out["is_dirty"].sum()
clean_count   = total_records - dirty_count

print(f"✅ Exported → {OUT_FILE}")
print(f"\n   Total records : {total_records}")
print(f"   Clean         : {clean_count}")
print(f"   Dirty         : {dirty_count}")
print(f"\n   Mutation breakdown:")
print(df_out[df_out["is_dirty"]]["mutation_category"].value_counts().to_string())

## Get OAUTH Token - Address Verification via REST API
Intent: Get OAUTH Token towards calling the RESTAPI using the key and secret.
Method: Generate token with baseencoding and get token

Why: Need OAUTH token for calling API

In [ ]:
import base64
import time

# ── Token manager — auto-refreshes on expiry ───────────────────────────────
class PreciselyAuth:
    def __init__(self, api_key, api_secret, token_url):
        self.api_key    = api_key
        self.api_secret = api_secret
        self.token_url  = token_url
        self._token     = None
        self._expires_at = 0

    def get_token(self):
        if time.time() < self._expires_at - 30:
            return self._token
        credentials = base64.b64encode(f"{self.api_key}:{self.api_secret}".encode()).decode()
        resp = requests.post(
            self.token_url,
            headers={
                "Authorization": f"Basic {credentials}",
                "Content-Type" : "application/x-www-form-urlencoded"
            },
            data={"grant_type": "client_credentials"}
        )
        resp.raise_for_status()
        data             = resp.json()
        self._token      = data["access_token"]
        self._expires_at = time.time() + int(data.get("expires_in", 3600))
        print(f"🔑 Token refreshed — expires in {data.get('expires_in', 3600)}s")
        return self._token

    def headers(self):
        return {
            "Authorization": f"Bearer {self.get_token()}",
            "Content-Type" : "application/json"
        }

auth = PreciselyAuth(PRECISELY_API_KEY, PRECISELY_API_SECRET, PRECISELY_TOKEN_URL)
# Test token acquisition
_ = auth.get_token()
print("✅ Auth ready")

## Address Verification via REST API
Intent: Establish a REST API baseline for address verification.
Method: Calls Precisely Address Verification REST API (/address/v1/transient/verify) on all records. Captures verify_status, corrected address fields, and match confidence. Saves to synthetic_250_restapi.csv.
Why: Provides a direct API comparison point against the MCP path.

In [ ]:
VERIFY_URL = "https://api.precisely.com/addressverification/v1/validatemailingaddress/results.json"

def verify_address(row):
    payload = {
        "options": {
            "OutputCasing": "M"
        },
        "Input": {
            "Row": [{
                "AddressLine1" : row["ADDRLINE1"],
                "AddressLine2" : row["ADDRLINE2"] if row["ADDRLINE2"] else "",
                "City"         : row["CITY"],
                "StateProvince": row["ADMIN1"],
                "PostalCode"   : row["POSTALCODE"],
                "Country"      : "USA"
            }]
        }
    }
    resp = requests.post(
        VERIFY_URL,
        headers=auth.headers(),
        json=payload,
        timeout=30
    )
    resp.raise_for_status()
    data = resp.json()
    try:
        out   = data["Output"][0]
        block = out.get("BlockAddress", "")

        # Compare input vs output to detect corrections
        in_street  = row["ADDRLINE1"].strip().upper()
        out_street = out.get("AddressLine1", "").strip().upper()
        in_zip     = row["POSTALCODE"].strip()
        out_zip    = out.get("PostalCode.Base", "").strip()
        in_city    = row["CITY"].strip().upper()
        out_city   = out.get("City", "").strip().upper()

        street_corrected = in_street != out_street
        zip_corrected    = in_zip    != out_zip
        city_corrected   = in_city   != out_city
        any_corrected    = street_corrected or zip_corrected or city_corrected

        status = "corrected" if any_corrected else "verified"

        return {
            "verify_status"    : status,
            "verified_address" : out.get("AddressLine1", ""),
            "verified_city"    : out.get("City", ""),
            "verified_state"   : out.get("StateProvince", ""),
            "verified_zip"     : out.get("PostalCode", ""),
            "verified_block"   : block,
            "street_corrected" : street_corrected,
            "zip_corrected"    : zip_corrected,
            "city_corrected"   : city_corrected,
        }
    except Exception as e:
        return {
            "verify_status"   : "error",
            "verified_address": "",
            "verified_city"   : "",
            "verified_state"  : "",
            "verified_zip"    : "",
            "verified_block"  : "",
            "street_corrected": False,
            "zip_corrected"   : False,
            "city_corrected"  : False,
        }

# ── Run on all 250 records ─────────────────────────────────────────────────
for col in ["verify_status","verified_address","verified_city","verified_state","verified_zip","verified_block"]:
    df_out[col] = ""
for col in ["street_corrected","zip_corrected","city_corrected"]:
    df_out[col] = False

total = len(df_out)
print(f"🚀 Verifying {total} records...")

for i, idx in enumerate(df_out.index):
    result = verify_address(df_out.loc[idx])
    for col, val in result.items():
        df_out.at[idx, col] = val
    if (i + 1) % 50 == 0 or (i + 1) == total:
        print(f"  [{i+1}/{total}] done")

print(f"\n✅ Verification complete")
print(df_out["verify_status"].value_counts().to_string())

df_out.to_csv(OUT_FILE, index=False)
print(f"\n💾 Saved → {OUT_FILE}")

## Address Verification via MCP
Intent: Run verification through the Precisely MCP server — the first step in building the enriched, trusted data track that the AI layer will reason over.
Method: Calls the Precisely MCP verify_address tool via local MCP server (http://127.0.0.1:8000/mcp). Captures mcp_verify_status, mcp_verified_address, mcp_verified_city, mcp_verified_state, mcp_verified_zip, mcp_verify_score, and per-field correction flags (mcp_street_corrected, mcp_zip_corrected, mcp_city_corrected). Saves progress to synthetic_250_mcp.csv.

LLM Role: After verification, Ollama (Mistral 7b) is called on every record that was address corrected (the dirty ones). The LLM receives the original dirty address, the Precisely-corrected address, and the property type — and produces a one-sentence underwriting impact statement stored in mcp_llm_review. This is a focused, constrained reasoning step: the LLM is not selecting tools or routing — it is interpreting what the correction means for an insurance underwriter.

Why: Per-field correction flags enable the AI layer to know not just that an address was corrected, but exactly which dimension was wrong.

In [ ]:
MCP_VERIFY_TOOL = "verify_address"

# ── Work on a fresh copy for the MCP pipeline ─────────────────────────────
df_mcp = df_out[[
    "PBKEY","FORMATTEDADDRESS","ADDRLINE1","ADDRLINE2","CITY",
    "ADMIN1","POSTALCODE","PROPTYPE","FIPS","LATITUDE","LONGITUDE",
    "is_dirty","mutation_category","mutation_description",
    "expected_api_outcome","llm_outcome_review"
]].copy()

# ── Init MCP verify columns ────────────────────────────────────────────────
for col in ["mcp_verify_status","mcp_verified_address","mcp_verified_city",
            "mcp_verified_state","mcp_verified_zip","mcp_llm_review"]:
    df_mcp[col] = ""
df_mcp["mcp_verify_score"] = 0
for col in ["mcp_street_corrected","mcp_zip_corrected","mcp_city_corrected"]:
    df_mcp[col] = False

def mcp_verify(row):
    addr = f"{row['ADDRLINE1']}, {row['CITY']}, {row['ADMIN1']} {row['POSTALCODE']}"
    resp = requests.post(
        MCP_BASE_URL,
        headers={
            "Accept"      : "application/json, text/event-stream",
            "Content-Type": "application/json"
        },
        json={
            "jsonrpc": "2.0",
            "id"     : 1,
            "method" : "tools/call",
            "params" : {
                "name"     : MCP_VERIFY_TOOL,
                "arguments": {"address": addr, "country": "USA"}
            }
        },
        timeout=15
    )
    resp.raise_for_status()
    data = resp.json()
    try:
        result  = data["result"]["structuredContent"]["responses"][0]["results"][0]
        address = result.get("address", {})
        score   = result.get("score", 0)

        out_street = address.get("formattedStreetAddress", "").strip().upper()
        out_city   = address.get("city", {}).get("longName", "").strip().upper()
        out_state  = address.get("admin1", {}).get("shortName", "").strip().upper()
        out_zip    = address.get("postalCode", "").strip()

        in_street  = row["ADDRLINE1"].strip().upper()
        in_city    = row["CITY"].strip().upper()
        in_zip     = row["POSTALCODE"].strip()

        street_corrected = in_street != out_street
        city_corrected   = in_city   != out_city
        zip_corrected    = in_zip    != out_zip
        status           = "corrected" if any([street_corrected, city_corrected, zip_corrected]) else "verified"

        return {
            "mcp_verify_status"    : status,
            "mcp_verified_address" : address.get("formattedStreetAddress", ""),
            "mcp_verified_city"    : address.get("city", {}).get("longName", ""),
            "mcp_verified_state"   : address.get("admin1", {}).get("shortName", ""),
            "mcp_verified_zip"     : address.get("postalCode", ""),
            "mcp_verify_score"     : score,
            "mcp_street_corrected" : street_corrected,
            "mcp_zip_corrected"    : zip_corrected,
            "mcp_city_corrected"   : city_corrected,
        }
    except Exception:
        return {
            "mcp_verify_status"    : "error",
            "mcp_verified_address" : "",
            "mcp_verified_city"    : "",
            "mcp_verified_state"   : "",
            "mcp_verified_zip"     : "",
            "mcp_verify_score"     : 0,
            "mcp_street_corrected" : False,
            "mcp_zip_corrected"    : False,
            "mcp_city_corrected"   : False,
        }

# ── Run verification on all 250 records ───────────────────────────────────
total = len(df_mcp)
print(f"🚀 MCP verify — {total} records...")

for i, idx in enumerate(df_mcp.index):
    result = mcp_verify(df_mcp.loc[idx])
    for col, val in result.items():
        df_mcp.at[idx, col] = val
    if (i + 1) % 50 == 0 or (i + 1) == total:
        print(f"  [{i+1}/{total}] done")

print(f"\n✅ MCP verification complete")
print(df_mcp["mcp_verify_status"].value_counts().to_string())

# ── LLM review on corrected records only ──────────────────────────────────
corrected_idx = df_mcp[df_mcp["mcp_verify_status"] == "corrected"].index.tolist()
print(f"\n🤖 LLM review — {len(corrected_idx)} corrected records using {OLLAMA_MODEL}...")

for i, idx in enumerate(corrected_idx):
    row    = df_mcp.loc[idx]
    prompt = (
        f"Original address  : {row['ADDRLINE1']}, {row['CITY']}, {row['ADMIN1']} {row['POSTALCODE']}\n"
        f"Corrected address : {row['mcp_verified_address']}, {row['mcp_verified_city']}, "
        f"{row['mcp_verified_state']} {row['mcp_verified_zip']}\n"
        f"Property type     : {row['PROPTYPE']}\n"
        f"Reply with JSON only: {{\"mcp_llm_review\": \"one sentence on the insurance underwriting impact of this correction\"}}"
    )
    try:
        resp   = requests.post(
            OLLAMA_URL,
            json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False},
            timeout=30
        )
        raw    = resp.json().get("response", "").strip()
        raw    = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
        start  = raw.find("{")
        end    = raw.rfind("}") + 1
        parsed = json.loads(raw[start:end])
        review = parsed.get("mcp_llm_review", "")
    except Exception as e:
        review = f"parse_error: {e}"

    df_mcp.at[idx, "mcp_llm_review"] = review

    if (i + 1) % 10 == 0 or (i + 1) == len(corrected_idx):
        print(f"  [{i+1}/{len(corrected_idx)}] done")

print(f"\n✅ LLM review complete")

# ── Save ───────────────────────────────────────────────────────────────────
MCP_FILE = DATA_DIR / "synthetic_250_mcp.csv"
df_mcp.to_csv(MCP_FILE, index=False)
print(f"\n💾 Saved → {MCP_FILE}")

# ── Preview ────────────────────────────────────────────────────────────────
print(df_mcp[df_mcp["mcp_verify_status"] == "corrected"][
    ["ADDRLINE1","mcp_verified_address","mcp_verify_status","mcp_llm_review"]
].head(5).to_string())

## Geocoding via REST API
Intent: Attempt geocoding through the direct REST path for baseline comparison.
Method: Calls Precisely Geocoding REST API (/geocode/v1/transient/geocode). Captures geo_status, geo_latitude, geo_longitude, geo_match_score.
Why: Establishes the REST geocode baseline. 
Note: REST geocode parsing encountered issues in this pipeline — results were incomplete. 

In [ ]:
GEOCODE_URL = "https://api.precisely.com/geocode/v1/basic/geocode"
BATCH_SIZE  = 100

# ── Null out PBKEY for corrected records ───────────────────────────────────
corrected_mask = df_out["verify_status"] == "corrected"
df_out.loc[corrected_mask, "PBKEY"] = ""
print(f"🔑 PBKEY cleared for {corrected_mask.sum()} corrected records")

# ── Init output columns ────────────────────────────────────────────────────
for col in ["geo_latitude","geo_longitude","geo_match_score","geo_result_code","geo_status"]:
    df_out[col] = ""

def build_address_entry(row):
    """Use verified fields if available, fall back to original."""
    return {
        "addressLine1": row["verified_address"] if row["verified_address"] else row["ADDRLINE1"],
        "areaName3"   : row["verified_city"]    if row["verified_city"]    else row["CITY"],
        "areaName1"   : row["verified_state"]   if row["verified_state"]   else row["ADMIN1"],
        "postCode1"   : row["verified_zip"]     if row["verified_zip"]     else row["POSTALCODE"],
        "country"     : "USA"
    }

def geocode_batch(batch_rows):
    payload = {
        "type": "ADDRESS",
        "preferences": {
            "maxReturnedCandidates" : 1,
            "returnAllCandidateInfo": "true",
            "fallbackToGeographic"  : "true",
            "fallbackToPostal"      : "true",
            "matchMode"             : "STANDARD",
            "clientCoordSysName"    : "EPSG:4326"
        },
        "addresses": [build_address_entry(row) for _, row in batch_rows]
    }
    resp = requests.post(
        GEOCODE_URL,
        headers=auth.headers(),
        json=payload,
        timeout=60
    )
    resp.raise_for_status()
    return resp.json().get("responses", [])

# ── Batch loop ─────────────────────────────────────────────────────────────
all_rows = list(df_out.iterrows())
total    = len(all_rows)
batches  = [all_rows[i:i+BATCH_SIZE] for i in range(0, total, BATCH_SIZE)]

print(f"\n🚀 Geocoding {total} records in {len(batches)} batches of {BATCH_SIZE}...")

for b_num, batch in enumerate(batches):
    responses = geocode_batch(batch)
    for pos, (idx, row) in enumerate(batch):
        try:
            candidate = responses[pos]["candidates"][0]
            coords    = candidate["geometry"]["coordinates"]
            custom    = candidate["address"].get("customFields", {})
            df_out.at[idx, "geo_latitude"]    = coords[1]
            df_out.at[idx, "geo_longitude"]   = coords[0]
            df_out.at[idx, "geo_match_score"] = custom.get("MATCH_SCORE", "")
            df_out.at[idx, "geo_result_code"] = custom.get("RESULT_CODE", "")
            df_out.at[idx, "geo_status"]      = "geocoded"
        except Exception:
            df_out.at[idx, "geo_status"]      = "failed"
    print(f"  Batch {b_num+1}/{len(batches)} done — {len(batch)} records")

print(f"\n✅ Geocoding complete")
print(df_out["geo_status"].value_counts().to_string())
print(f"\nMatch score distribution (dirty vs clean):")
print(df_out.groupby(["is_dirty","geo_match_score"]).size().to_string())


df_out.to_csv(OUT_FILE, index=False)
print(f"\n💾 Saved → {OUT_FILE}")

## Geocoding via MCP
Intent: Geocode all 250 verified records to building-level precision and assign PreciselyID — the persistent location anchor that makes the enriched data track linkable and AI-ready.
Method: Calls the Precisely MCP geocode tool. Captures mcp_geo_status, mcp_geo_pb_key (PreciselyID), mcp_geo_precision, mcp_geo_latitude, mcp_geo_longitude, mcp_geo_match_score. Records with score 100 are ADDRESS_POINT precision (building level). Records with score 50 are too corrupt to resolve. 227/250 records received a PBKEY.

Why: PBKEY is what separates a coordinate from a trusted location identity.

In [ ]:
MCP_GEOCODE_TOOL = "geocode"

# ── Init MCP geocode columns ───────────────────────────────────────────────
for col in ["mcp_geo_status","mcp_geo_pb_key","mcp_geo_precision"]:
    df_mcp[col] = ""
df_mcp["mcp_geo_latitude"]    = 0.0
df_mcp["mcp_geo_longitude"]   = 0.0
df_mcp["mcp_geo_match_score"] = 0

def mcp_geocode(row):
    # Use MCP verified address if available, else original
    addr  = row["mcp_verified_address"] if row["mcp_verified_address"] else row["ADDRLINE1"]
    city  = row["mcp_verified_city"]    if row["mcp_verified_city"]    else row["CITY"]
    state = row["mcp_verified_state"]   if row["mcp_verified_state"]   else row["ADMIN1"]
    zcode = row["mcp_verified_zip"]     if row["mcp_verified_zip"]     else row["POSTALCODE"]

    resp = requests.post(
        MCP_BASE_URL,
        headers={
            "Accept"      : "application/json, text/event-stream",
            "Content-Type": "application/json"
        },
        json={
            "jsonrpc": "2.0",
            "id"     : 1,
            "method" : "tools/call",
            "params" : {
                "name"     : MCP_GEOCODE_TOOL,
                "arguments": {
                    "address": f"{addr}, {city}, {state} {zcode}",
                    "country": "USA"
                }
            }
        },
        timeout=15
    )
    resp.raise_for_status()
    data = resp.json()
    try:
        result  = data["result"]["structuredContent"]["responses"][0]["results"][0]
        coords  = result["location"]["feature"]["geometry"]["coordinates"]
        custom  = result.get("customFields", {})
        return {
            "mcp_geo_latitude"   : coords[1],
            "mcp_geo_longitude"  : coords[0],
            "mcp_geo_match_score": result.get("score", 0),
            "mcp_geo_pb_key"     : custom.get("PB_KEY", ""),
            "mcp_geo_precision"  : result.get("location", {}).get("explanation", {}).get("type", ""),
            "mcp_geo_status"     : "geocoded"
        }
    except Exception:
        return {
            "mcp_geo_latitude"   : 0.0,
            "mcp_geo_longitude"  : 0.0,
            "mcp_geo_match_score": 0,
            "mcp_geo_pb_key"     : "",
            "mcp_geo_precision"  : "",
            "mcp_geo_status"     : "failed"
        }

# ── Run on all 250 records ─────────────────────────────────────────────────
total = len(df_mcp)
print(f"🚀 MCP geocode — {total} records...")

for i, idx in enumerate(df_mcp.index):
    result = mcp_geocode(df_mcp.loc[idx])
    for col, val in result.items():
        df_mcp.at[idx, col] = val
    if (i + 1) % 50 == 0 or (i + 1) == total:
        print(f"  [{i+1}/{total}] done")

print(f"\n✅ MCP geocoding complete")
print(df_mcp["mcp_geo_status"].value_counts().to_string())
print(f"\nMatch score distribution (dirty vs clean):")
print(df_mcp.groupby(["is_dirty","mcp_geo_match_score"]).size().to_string())
print(f"\nPB_KEY coverage: {(df_mcp['mcp_geo_pb_key'] != '').sum()} / {total}")
print(f"ADDRESS_POINT precision: {(df_mcp['mcp_geo_precision'] == 'ADDRESS_POINT').sum()} / {total}")


df_mcp.to_csv(DATA_DIR / "synthetic_250_mcp.csv", index=False)
print(f"\n💾 Saved → {DATA_DIR / 'synthetic_250_mcp.csv'}")
print(f"\n   Total records        : {len(df_mcp)}")
print(f"   Clean                : {(df_mcp['is_dirty'] == False).sum()}")
print(f"   Dirty                : {df_mcp['is_dirty'].sum()}")
print(f"   MCP verified         : {(df_mcp['mcp_verify_status'] == 'verified').sum()}")
print(f"   MCP corrected        : {(df_mcp['mcp_verify_status'] == 'corrected').sum()}")
print(f"   Geocoded             : {(df_mcp['mcp_geo_status'] == 'geocoded').sum()}")
print(f"   PB_KEY coverage      : {(df_mcp['mcp_geo_pb_key'] != '').sum()}")
print(f"   ADDRESS_POINT        : {(df_mcp['mcp_geo_precision'] == 'ADDRESS_POINT').sum()}")



## Address Intelligence Report - Before/After
Intent: Produce a pipeline results.
Method: Provides verification rates, correction rates, geocode coverage, and match score distribution.
Why: This is the "before/after" quantifying how much dirty data the pipeline catches and corrects.

In [24]:

print("=" * 65)
print("  ADDRESS INTELLIGENCE REPORT - BEFORE/AFTER")
print("=" * 65)


# ── 1. Match rates ─────────────────────────────────────────────────────────
rest_matched = (df_out["verify_status"].isin(["verified","corrected"])).sum()
mcp_matched  = (df_mcp["mcp_verify_status"].isin(["verified","corrected"])).sum()

print(f"\n1. MATCH RATES")
print(f"   {'Pipeline':<20} {'Matched':>10} {'Total':>8} {'Rate':>8}")
print(f"   {'-'*46}")
print(f"   {'REST':<20} {rest_matched:>10} {total:>8} {rest_matched/total*100:>7.1f}%")
print(f"   {'MCP':<20} {mcp_matched:>10} {total:>8} {mcp_matched/total*100:>7.1f}%")

# ── 2. Standardization improvements ───────────────────────────────────────
rest_corrected = (df_out["verify_status"] == "corrected").sum()
mcp_corrected  = (df_mcp["mcp_verify_status"] == "corrected").sum()

print(f"\n2. STANDARDIZATION IMPROVEMENTS (dirty {dirty_count} records)")
print(f"   {'Metric':<30} {'REST':>8} {'MCP':>8}")
print(f"   {'-'*46}")
print(f"   {'Records corrected':<30} {rest_corrected:>8} {mcp_corrected:>8}")
print(f"   {'Correction rate':<30} {rest_corrected/dirty_count*100:>7.1f}% {mcp_corrected/dirty_count*100:>7.1f}%")
print(f"   {'Street standardized':<30} {df_out['street_corrected'].sum():>8} {df_mcp['mcp_street_corrected'].sum():>8}")
print(f"   {'ZIP standardized':<30} {df_out['zip_corrected'].sum():>8} {df_mcp['mcp_zip_corrected'].sum():>8}")
print(f"   {'City standardized':<30} {df_out['city_corrected'].sum():>8} {df_mcp['mcp_city_corrected'].sum():>8}")

# ── 3. Coordinate coverage ─────────────────────────────────────────────────
rest_geocoded = (df_out["geo_status"] == "geocoded").sum()
mcp_geocoded  = (df_mcp["mcp_geo_status"] == "geocoded").sum()
mcp_exact     = (df_mcp["mcp_geo_match_score"] == 100).sum()
mcp_pbkey     = (df_mcp["mcp_geo_pb_key"] != "").sum()
mcp_addrpt    = (df_mcp["mcp_geo_precision"] == "ADDRESS_POINT").sum()

print(f"\n3. COORDINATE COVERAGE")
print(f"   {'Metric':<30} {'REST':>8} {'MCP':>8}")
print(f"   {'-'*46}")
print(f"   {'Records geocoded':<30} {rest_geocoded:>8} {mcp_geocoded:>8}")
print(f"   {'Coverage rate':<30} {rest_geocoded/total*100:>7.1f}% {mcp_geocoded/total*100:>7.1f}%")
print(f"   {'Score 100 — exact match':<30} {'N/A':>8} {mcp_exact:>8}")
print(f"   {'ADDRESS_POINT precision':<30} {'N/A':>8} {mcp_addrpt:>8}")
print(f"   {'PreciselyID assigned':<30} {'N/A':>8} {mcp_pbkey:>8}")

# ── 4. Undetected dirty data analysis ───────────────────────────────────────
undetected = df_mcp[df_mcp["is_dirty"] & (df_mcp["mcp_verify_status"] == "verified")]
undetected_cats = undetected["mutation_category"].value_counts()

print(f"\n4. UNDETECTED MUTATIONS — VERIFICATION BLIND SPOTS")
print(f"   {'Category':<30} {'Count':>8} {'Reason'}")
print(f"   {'-'*62}")
reasons = {
    "missing_unit"      : "Unit omission — address still routes without unit",
    "street_typo"       : "Typo close enough to resolve to a valid street",
    "flood_zip_mismatch": "Inland ZIP matched a valid address at that location",
    "zip_city_mismatch" : "Valid ZIP/city pairing accepted as deliverable",
    "unresolvable"      : "Address too corrupt — no candidate found"
}
for cat, count in undetected_cats.items():
    print(f"   {cat:<30} {count:>8}  {reasons.get(cat, '')}")

print(f"\n   ⚠️  Missing unit numbers are the highest risk blind spot.")
print(f"   A unit-less address on a multi-occupancy building")
print(f"   passes verification but cannot be tied to a specific")
print(f"   policyholder — creating premium mispricing exposure.")


total       = len(df_out)
dirty_count = df_out["is_dirty"].sum()
clean_count = total - dirty_count

print("=" * 65)

  ADDRESS INTELLIGENCE REPORT - BEFORE/AFTER


NameError: name 'df_out' is not defined

## Location Enrichment Layer
Intent: Append 4 insurance-relevant location attributes to every verified record — deliberately feeding the AI layer in Cell 13 the correlated context it needs to reason about risk, not just report data.
Method: Calls four Precisely MCP enrichment tools sequentially per record:

get_flood_risk_by_address — FEMA flood zone, distance to 100yr/500yr flood boundary, elevation. Tells the AI whether the property sits in a regulated flood zone and how close the risk boundary is.
get_property_data — Building type (residential/commercial/mixed) and total building area in sq ft. Gives the AI the physical structure context needed to assess replacement cost risk.
get_property_fire_risk — Distance to nearest fire station in miles, AM peak and overnight drive times. Gives the AI response time data that directly influences fire risk scoring and ISO protection class reasoning.
get_demographics — PSYTE geodemographic segment, household income tier, average household income, average home value, average rent. Gives the AI neighborhood context to reason about underinsurance relative to property value, and fraud risk patterns.

Why: Enrichment transforms a verified address into a decision-ready location profile.

In [ ]:
# ============================================================
# Location Enrichment Layer
# Precisely APIs sequentially for each verified record
# APIs: Flood Risk, Property Attributes, Fire Risk, Demographics
# Runs against df_mcp (250 verified + geocoded records)
# ============================================================

import json
import time
import requests
import pandas as pd
from pathlib import Path
from tqdm import tqdm

BASE_DIR = Path().resolve().parent
DATA_DIR = BASE_DIR / "data"

# ── Load verified/geocoded dataset ───────────────────────────
MCP_FILE = DATA_DIR / "synthetic_250_mcp.csv"
df_mcp   = pd.read_csv(MCP_FILE)
print(f"✅ Loaded {len(df_mcp)} records from {MCP_FILE.name}")



MCP_HEADERS = {
    "Accept"      : "application/json, text/event-stream",
    "Content-Type": "application/json"
}

# ── MCP tool caller ───────────────────────────────────────────
def call_mcp_tool(tool_name, params, retries=2):
    """Call a Precisely MCP tool by name with given params."""
    payload = {
        "jsonrpc": "2.0",
        "id"     : 1,
        "method" : "tools/call",
        "params" : {"name": tool_name, "arguments": params}
    }
    for attempt in range(retries + 1):
        try:
            r = requests.post(MCP_BASE_URL, headers=MCP_HEADERS, json=payload, timeout=15)
            r.raise_for_status()
            result = r.json()
            if "result" in result:
                content = result["result"].get("content", [])
                if content and content[0].get("type") == "text":
                    return json.loads(content[0]["text"])
            return {}
        except Exception as e:
            if attempt < retries:
                time.sleep(1)
            else:
                return {"error": str(e)}

# ── Address builder helper ────────────────────────────────────
def build_address(row):
    """Build a clean address string from verified MCP fields."""
    addr  = row.get("mcp_verified_address") or row.get("ADDRLINE1", "")
    city  = row.get("mcp_verified_city")    or row.get("CITY", "")
    state = row.get("mcp_verified_state")   or row.get("ADMIN1", "")
    zipp  = row.get("mcp_verified_zip")     or row.get("POSTALCODE", "")
    return f"{addr}, {city}, {state} {zipp}".strip(", ")

# ── Per-record enrichment function ────────────────────────────
def safe_get(data, *keys, default=""):
    """Safely traverse nested dict keys."""
    for key in keys:
        if isinstance(data, dict):
            data = data.get(key, default)
        elif isinstance(data, list):
            data = data[0] if data else default
        else:
            return default
    return data if data is not None else default

def enrich_record(row):
    address = build_address(row)
    result  = {}

    # ── 1. Flood Risk ─────────────────────────────────────────
    flood     = call_mcp_tool("get_flood_risk_by_address", {"address": address})
    flood_rec = safe_get(flood, "data", "getByAddress", "addresses", "data", 0,
                         "floodRisk", "data", 0)
    result["enrich_flood_zone"]      = safe_get(flood_rec, "floodZone")
    result["enrich_flood_panel_no"]  = safe_get(flood_rec, "femaMapPanelIdentifier")
    result["enrich_flood_dist_100yr"]= safe_get(flood_rec, "year100FloodZoneDistanceFeet")
    result["enrich_flood_dist_500yr"]= safe_get(flood_rec, "year500FloodZoneDistanceFeet")
    result["enrich_flood_elevation"] = safe_get(flood_rec, "addressLocationElevationFeet")

    # ── 2. Property Data (buildings) ──────────────────────────
    prop      = call_mcp_tool("get_property_data", {"address": address})
    bldg_rec  = safe_get(prop, "data", "getByAddress", "buildings", "data", 0)
    result["enrich_building_type"]     = safe_get(bldg_rec, "buildingType", "description")
    result["enrich_building_area_sqft"]= safe_get(bldg_rec, "buildingArea")

    # ── 3. Fire Risk ──────────────────────────────────────────
    fire     = call_mcp_tool("get_property_fire_risk", {"address": address})
    fire_rec = safe_get(fire, "data", "getByAddress", "addresses", "data", 0,
                        "propertyFireRisk", "data", 0)
    result["enrich_fire_station_dist_mi"]    = safe_get(fire_rec, "firestation1DriveDistanceMiles")
    result["enrich_fire_drivetime_peak_min"] = safe_get(fire_rec, "firestation1DrivetimeAMPeakMinutes")
    result["enrich_fire_drivetime_night_min"]= safe_get(fire_rec, "firestation1DrivetimeNightMinutes")

    # ── 4. Demographics ───────────────────────────────────────
    demo      = call_mcp_tool("get_demographics", {"address": address})
    demo_base = safe_get(demo, "data", "getByAddress", "addresses", "data", 0)
    psyte_rec = safe_get(demo_base, "psyteGeodemographics", "data", 0)
    gview_rec = safe_get(demo_base, "groundView", "data", 0)

    result["enrich_segment"]         = safe_get(psyte_rec, "PSYTESegmentCode", "description")
    result["enrich_income_tier"]      = safe_get(psyte_rec, "householdIncomeVariable", "description")
    result["enrich_avg_income"]       = safe_get(gview_rec, "averageHouseholdIncome")
    result["enrich_avg_home_value"]   = safe_get(gview_rec, "averageHomeValue")
    result["enrich_avg_rent"]         = safe_get(gview_rec, "averageRent")

    return result

# ── Run enrichment across all 250 records ─────────────────────
print("🔄 Running enrichment on all records — this may take a few minutes...")
print(f"   Total records: {len(df_mcp)}\n")

enrichment_results = []
for idx, row in tqdm(df_mcp.iterrows(), total=len(df_mcp), desc="Enriching"):
    enrichment_results.append(enrich_record(row))
    time.sleep(0.2)   # polite rate limiting

df_enriched = pd.concat(
    [df_mcp.reset_index(drop=True), pd.DataFrame(enrichment_results)],
    axis=1
)

# ── Summary ───────────────────────────────────────────────────
enrich_cols = [c for c in df_enriched.columns if c.startswith("enrich_")]
coverage    = {col: (df_enriched[col] != "").sum() for col in enrich_cols}

print("\n✅ Enrichment complete")
print(f"   Records enriched : {len(df_enriched)}")
print("\n   Coverage by field:")
for col, count in coverage.items():
    print(f"   {col:<35} {count} / {len(df_enriched)}")

# ── Save enriched dataset ─────────────────────────────────────
ENRICHED_FILE = DATA_DIR / "synthetic_250_enriched.csv"
df_enriched.to_csv(ENRICHED_FILE, index=False)
print(f"\n💾 Saved → {ENRICHED_FILE}")

## LLM Creating Narratives for Basic data & Enriched data

In [32]:
import math


BASE_DIR = Path().resolve().parent
DATA_DIR = BASE_DIR / "data"

# ── Load verified/geocoded dataset ───────────────────────────
MCP_FILE = DATA_DIR / "synthetic_250_enriched.csv"
df_mcp   = pd.read_csv(MCP_FILE)
print(f"✅ Loaded {len(df_mcp)} records from {MCP_FILE.name}")
SCORED_FILE = DATA_DIR / "synthetic_250_scored.csv"

# ── Init narrative columns ─────────────────────────────────────────────────
for col in ["basic_narrative", "enhanced_narrative"]:
    df_mcp[col] = ""

# ── Prompts ────────────────────────────────────────────────────────────────
BASIC_PROMPT = """You are an insurance underwriting AI. Write a 2-3 sentence underwriting assessment for this property based only on the verified address and property type.

Address      : {address}
Property type: {proptype}

Be specific about what you can and cannot determine from the address alone.
Reply with ONLY the narrative text, no JSON, no labels."""

ENHANCED_PROMPT = """You are an insurance underwriting AI. Write a 2-3 sentence underwriting assessment for this property using the verified address and full location intelligence data.

Address          : {address}
Property type    : {proptype}
Flood zone       : {flood_zone}
Dist to 100yr    : {flood_dist_100yr} ft
Dist to 500yr    : {flood_dist_500yr} ft
Elevation        : {flood_elevation} ft
Building type    : {building_type}
Building sqft    : {building_sqft}
Fire station     : {fire_dist} mi
Fire drive peak  : {fire_peak} min
Fire drive night : {fire_night} min
Neighborhood     : {segment}
Income tier      : {income_tier}
Avg income       : {avg_income}
Avg home value   : {avg_home_value}
Avg rent         : {avg_rent}

Lead with the dominant risk factor. Be specific about what the enrichment data reveals.
Reply with ONLY the narrative text, no JSON, no labels."""

# ── Helper ─────────────────────────────────────────────────────────────────
def fmt(val, sentinel=-999999.0):
    if val is None or (isinstance(val, float) and (math.isnan(val) or val == sentinel)):
        return "unavailable"
    return str(val)

def llm_narrative(prompt, timeout=45):
    resp = requests.post(
        OLLAMA_URL,
        json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False},
        timeout=timeout
    )
    raw = resp.json().get("response", "").strip()
    return re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()

# ── Narrative loop ─────────────────────────────────────────────────────────
total = len(df_mcp)
print(f"🚀 Generating narratives — {total} records using {OLLAMA_MODEL}...")

for i, idx in enumerate(df_mcp.index):
    row           = df_mcp.loc[idx]
    verified_addr = f"{row['mcp_verified_address'] or row['ADDRLINE1']}, {row['mcp_verified_city'] or row['CITY']}, {row['mcp_verified_state'] or row['ADMIN1']} {row['mcp_verified_zip'] or row['POSTALCODE']}"

    # Basic narrative
    try:
        df_mcp.at[idx, "basic_narrative"] = llm_narrative(BASIC_PROMPT.format(
            address  = verified_addr,
            proptype = row["PROPTYPE"]
        ))
    except Exception as e:
        df_mcp.at[idx, "basic_narrative"] = f"error: {e}"

    # Enhanced narrative
    try:
        df_mcp.at[idx, "enhanced_narrative"] = llm_narrative(ENHANCED_PROMPT.format(
            address         = verified_addr,
            proptype        = row["PROPTYPE"],
            flood_zone      = fmt(row["enrich_flood_zone"]),
            flood_dist_100yr= fmt(row["enrich_flood_dist_100yr"]),
            flood_dist_500yr= fmt(row["enrich_flood_dist_500yr"]),
            flood_elevation = fmt(row["enrich_flood_elevation"]),
            building_type   = fmt(row["enrich_building_type"]),
            building_sqft   = fmt(row["enrich_building_area_sqft"]),
            fire_dist       = fmt(row["enrich_fire_station_dist_mi"]),
            fire_peak       = fmt(row["enrich_fire_drivetime_peak_min"]),
            fire_night      = fmt(row["enrich_fire_drivetime_peak_min"]),
            segment         = fmt(row["enrich_segment"]),
            income_tier     = fmt(row["enrich_income_tier"]),
            avg_income      = fmt(row["enrich_avg_income"]),
            avg_home_value  = fmt(row["enrich_avg_home_value"]),
            avg_rent        = fmt(row["enrich_avg_rent"])
        ))
    except Exception as e:
        df_mcp.at[idx, "enhanced_narrative"] = f"error: {e}"

    if (i + 1) % 25 == 0 or (i + 1) == total:
        print(f"  [{i+1}/{total}] done")

# ── Save ───────────────────────────────────────────────────────────────────
df_mcp.to_csv(SCORED_FILE, index=False)
print(f"\n✅ Narratives complete → {SCORED_FILE}")

# ── Preview ────────────────────────────────────────────────────────────────
sample = df_mcp[df_mcp["basic_narrative"] != ""].iloc[0]
print(f"\nSample — {sample['ADDRLINE1']}")
print(f"\nBasic:\n{sample['basic_narrative']}")
print(f"\nEnhanced:\n{sample['enhanced_narrative']}")

✅ Loaded 250 records from synthetic_250_enriched.csv
🚀 Generating narratives — 250 records using mistral:7b...
  [25/250] done
  [50/250] done
  [75/250] done
  [100/250] done
  [125/250] done
  [150/250] done
  [175/250] done
  [200/250] done
  [225/250] done
  [250/250] done

✅ Narratives complete → /Users/senthilpanchatcharam/projects/address-intelligence-prototype/data/synthetic_250_scored.csv

Sample — 500 W 30TH ST APT 15M

Basic:
Based on the provided address and property type "R," which typically denotes a rental property in New York City, I can infer that this is a high-rise apartment located in Manhattan's Chelsea neighborhood (500 W 30th St). However, without additional data, I cannot assess the property's condition, risk factors, or insurance history, which are essential components for accurate underwriting. Further investigation would be required to provide a comprehensive evaluation of this property's risk profile and appropriate insurance coverage.

Enhanced:
The primary

In [37]:

missing = df_mcp['enhanced_narrative'].isna().sum()
empty = (df_mcp['enhanced_narrative'] == '').sum()
print(f"NaN: {missing}, Empty string: {empty}, Total: {missing + empty}")


NaN: 0, Empty string: 0, Total: 0
